<a href="https://colab.research.google.com/github/gustavobraga-byte/PesquisAI/blob/main/PesquisAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬 PesquisAI — Agente de IA para Pesquisadores Científicos

O **PesquisAI** é um assistente de inteligência artificial executado diretamente no Google Colab, projetado para automatizar o fluxo de trabalho acadêmico de pesquisadores brasileiros — desde a revisão bibliográfica até a preparação para a submissão de artigos.

### 🛡️ Vantagens
* **100% Gratuito:** Executado na infraestrutura em nuvem do Google.
* **Sem Instalação:** Pronto para uso, direto no seu navegador.
* **Foco Nacional:** Integrado com ferramentas de dados brasileiras (**IBGE** e **DataSUS**).
* **Base Tecnológica:** Agente de IA que foi desenvolvido sob a arquitetura do **OpenCode**.

---

### 🚀 Como Iniciar

1. No menu superior, clique em **Ambiente de execução** ➡️ **Executar tudo** (ou pressione `Ctrl + F9`).
2. Aguarde cerca de **2 minutos** enquanto as dependências e *skills* são configuradas.
3. Role até o final da página e clique nos botões azuis:
    * **"🤖 Abrir o PesquisAI"** para iniciar a interface.
    * **"📁 Abrir a pasta de trabalho no Google Drive"** para acessar sua pasta de arquivos.

---

## ℹ️ Informações Importantes

*   **Navegador Preferencial:** Para uma experiência ideal, recomenda-se a utilização do **Google Chrome**.
*   **Conexão com a Internet:** Uma conexão estável com a internet é essencial para o download de dependências e a comunicação com as APIs.
*   **Google Drive:** O acesso ao Google Drive é necessário para salvar arquivos e gerenciar o diretório de trabalho do PesquisAI. <br>
⚠️ Os arquivos finais eventualamente gerados pelo agente são salvos nessa pasta ⚠️
*   **Permissões:** Certifique-se de conceder todas as permissões solicitadas pelo Google Colab, especialmente para acesso ao Google Drive, para que o assistente possa operar corretamente.

In [ ]:
!curl -fsSL https://opencode.ai/install | bash

In [ ]:
# Precisa disso para que o recurso de selecionar e copiar funcione
!sudo apt-get update && sudo apt-get install -y xclip xsel

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh

In [ ]:
!# Instalando a minha Skill do IBGE
!git clone https://github.com/gustavobraga-byte/Skill-IBGE.git ~/.agents/skills/ibge-br

In [ ]:
# Instalando a minha skill do OpenDataSUS
# Baixe o repositório
!git clone https://github.com/gustavobraga-byte/Skill-DataSus.git /tmp/opendatasus
# Copie para o diretório de skills
!cp -r /tmp/opendatasus ~/.agents/skills/

In [ ]:
# Instalando as skills do Kdense para pesquisa
# Baixe o repositório
!git clone https://github.com/K-Dense-AI/scientific-agent-skills.git /tmp/scientific
# Copie para o diretório de skills
!cp -r /tmp/scientific/scientific-skills ~/.agents/skills/

In [ ]:
# Instalando o Harness
# Baixe o repositório
!git clone https://github.com/gustavobraga-byte/PesquisAI.git /tmp/harness
# Listar o conteúdo para depuração
!ls -l /tmp/harness
# Copie para o diretório de skills
!cp /tmp/harness/AGENTS.md /content

In [ ]:
# Criando a pasta de trabalho no google drive, as permissões são necessárias

from google.colab import drive, auth
from IPython.display import HTML, display
from googleapiclient.discovery import build
import os

# Define the target folder in Google Drive and its local mount point
drive_folder_path = 'My Drive/PesquisAI'

# 1. Conecta o Google Drive
try:
    drive.mount('/content/drive', force_remount=True)
    # Correctly define caminho_pasta to reflect the actual mounted path for 'My Drive/PesquisAI'
    caminho_pasta = os.path.join('/content/drive', drive_folder_path)
    print(f"Drive montado. Caminho da pasta '{drive_folder_path}' definido como '{caminho_pasta}'.")

    # Garante que a pasta de trabalho exista no Drive montado. Se não existir, a cria.
    if not os.path.exists(caminho_pasta):
        print(f"Criando a pasta de trabalho: '{caminho_pasta}'")
        os.makedirs(caminho_pasta, exist_ok=True)
    else:
        print(f"Pasta de trabalho '{caminho_pasta}' já existe.")

    # Aprimoramento: Mudar o diretório de trabalho atual para a pasta do PesquisAI.
    # Isso garante que quaisquer operações de arquivo subsequentes (e.g., criação de arquivos, leitura)
    # sejam realizadas DENTRO desta pasta por padrão, a menos que um caminho absoluto diferente seja especificado.
    os.chdir(caminho_pasta)
    print(f"Diretório de trabalho alterado para: {os.getcwd()}")

    # Nota importante sobre permissões:
    # A montagem do Google Drive (`drive.mount`) concede acesso a todo o seu Google Drive
    # ao ambiente do Colab. No entanto, este script está configurado para operar
    # primariamente dentro da pasta especificada ('My Drive/PesquisAI'),
    # e qualquer operação de arquivo explícita ou API do Drive será direcionada a ela.
    # A autenticação do usuário para a API do Drive (build('drive', 'v3')) também opera
    # sob as permissões da sua conta do Google, mas as consultas são direcionadas.

except Exception as e:
    print(f"Erro ao montar o Google Drive ou configurar a pasta: {e}.")

# Autentica a API do Drive com a conta atual do Colab
auth.authenticate_user()

# CORREÇÃO: Usar 'v3' em vez de '3'
service = build('drive', 'v3')

# 2. A criação da pasta local já é tratada por `os.makedirs` acima.
# A API do Drive será usada para buscar o ID da pasta e gerar o link.

# 3. Busca o ID web real da pasta no Google Drive
url_direta = "https://drive.google.com"  # Link reserva caso falhe
try:
    # A query para a API do Drive está correta, buscando a pasta pelo nome
    query = f"name = '{drive_folder_path.split('/')[-1]}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
    resultado = service.files().list(q=query, fields="files(id)").execute()
    arquivos_drive = resultado.get('files', [])

    if arquivos_drive:
        # Acessar o primeiro item da lista mapeada
        id_pasta = arquivos_drive[0]['id']
        url_direta = f"https://drive.google.com/drive/folders/{id_pasta}" # Formato de URL corrigido
        print("Link direto para a pasta gerado com sucesso!")
    else:
        # Se a pasta foi criada via os.makedirs, a API do Drive pode não refleti-la imediatamente.
        print(f"Pasta '{drive_folder_path.split('/')[-1]}' não encontrada via Drive API. Pode ter sido criada localmente. Usando link geral do Meu Drive.")
except Exception as e:
    print(f"Não foi possível mapear o link direto ({e}). Usando link geral do Meu Drive.")



In [ ]:
import os
import subprocess
import time
import threading
from http.server import SimpleHTTPRequestHandler, HTTPServer
from google.colab import output
from IPython.display import display, HTML

# 0. INSTALAÇÃO OBRIGATÓRIA (Garante que vai funcionar em qualquer máquina nova)
print("📦 Instalando os paranauê que precisa...")
!apt-get update -qq && apt-get install -y -qq ttyd

# 1. Derruba qualquer terminal antigo para liberar a porta 8000
!pkill -f ttyd
!pkill -f "python3.*8001"
time.sleep(0.5)

# 2. Inicia o ttyd injetando a variável que desativa o bloqueio de mouse do OpenCode
print("🚀 Iniciando o terminal para o PesquisAI..")

# Cria uma cópia do ambiente do sistema e adiciona a flag de desativação
env_config = os.environ.copy()
env_config["OPENCODE_EXPERIMENTAL_DISABLE_COPY_ON_SELECT"] = "1"

subprocess.Popen(
    ["ttyd", "-p", "8000", "bash", "-i", "-c", "opencode; exec bash"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env=env_config  # Aplica o ambiente modificado aqui
)

time.sleep(2) # Aguarda 2 segundos para o terminal subir totalmente


# 3. Captura a URL real do proxy da porta 8000 do Colab
terminal_url = output.eval_js('google.colab.kernel.proxyPort(8000)')
banner_url = output.eval_js('google.colab.kernel.proxyPort(8001)')

# 4. URL do Drive (vinda da célula anterior ou fallback)
drive_url = globals().get('url_direta', 'https://drive.google.com/drive/my-drive')

# 5. Cria a página wrapper com banner + terminal embedado
wrapper_html = f'''<!DOCTYPE html>
<html lang="pt-BR">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>PesquisAI</title>
    <style>
        * {{ margin: 0; padding: 0; box-sizing: border-box; }}
        html, body {{ height: 100%; overflow: hidden; font-family: "Segoe UI", Tahoma, Geneva, Verdana, sans-serif; background: #0a0a0a; }}
        #banner {{
            position: fixed;
            top: 0; left: 0; right: 0;
            height: 42px;
            background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%);
            display: flex;
            align-items: center;
            justify-content: space-between;
            padding: 0 14px;
            z-index: 9999;
            box-shadow: 0 2px 12px rgba(0,0,0,0.5);
            border-bottom: 1px solid rgba(0, 210, 255, 0.25);
        }}
        .brand {{
            display: flex;
            align-items: center;
            gap: 7px;
            font-size: 15px;
            font-weight: 700;
            color: #fff;
        }}
        .brand .highlight {{
            background: linear-gradient(90deg, #007bff, #00d2ff);
            -webkit-background-clip: text;
            -webkit-text-fill-color: transparent;
            background-clip: text;
        }}
        .drive-btn {{
            display: inline-flex;
            align-items: center;
            gap: 5px;
            background: rgba(255,255,255,0.07);
            color: #bbb;
            border: 1px solid rgba(255,255,255,0.12);
            padding: 5px 12px;
            font-size: 11.5px;
            font-weight: 600;
            border-radius: 18px;
            text-decoration: none;
            transition: all 0.2s ease;
            cursor: pointer;
        }}
        .drive-btn:hover {{
            background: rgba(0, 123, 255, 0.2);
            border-color: rgba(0, 123, 255, 0.45);
            color: #fff;
            transform: translateY(-1px);
            box-shadow: 0 2px 8px rgba(0, 123, 255, 0.25);
        }}
        #terminal-frame {{
            position: absolute;
            top: 42px; left: 0; right: 0; bottom: 0;
            width: 100%;
            height: calc(100% - 42px);
            border: none;
        }}
    </style>
</head>
<body>
    <div id="banner">
        <div class="brand">
            <span>👨‍🔬</span>
            <span>🤖</span>
            <span class="highlight">PesquisAI</span>
        </div>
        <a href="{drive_url}" target="_blank" class="drive-btn">
            📁 Abrir pasta PesquisaAI no Drive
        </a>
    </div>
    <iframe id="terminal-frame" src="{terminal_url}" allow="clipboard-read; clipboard-write"></iframe>
</body>
</html>'''

# Salva e serve o wrapper na porta 8001
os.makedirs('/tmp/pesquisai-wrapper', exist_ok=True)
with open('/tmp/pesquisai-wrapper/index.html', 'w') as f:
    f.write(wrapper_html)

class QuietHandler(SimpleHTTPRequestHandler):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, directory='/tmp/pesquisai-wrapper', **kwargs)
    def log_message(self, fmt, *args):
        pass

def serve_wrapper():
    HTTPServer(('0.0.0.0', 8001), QuietHandler).serve_forever()

threading.Thread(target=serve_wrapper, daemon=True).start()
time.sleep(1)

# 6. Botão que abre a página com banner + terminal
botao_html = f'''
<div style="margin: 10px 0;">
    <a href="{banner_url}" target="_blank" style="text-decoration: none;">
        <button style="
            background: linear-gradient(135deg, #007bff, #00d2ff);
            border: none;
            color: white;
            padding: 14px 28px;
            font-size: 16px;
            font-weight: bold;
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            border-radius: 50px;
            cursor: pointer;
            box-shadow: 0 4px 15px rgba(0, 123, 255, 0.3);
            transition: all 0.3s ease;
            display: inline-flex;
            align-items: center;
            gap: 10px;
        " onmouseover="this.style.transform='scale(1.03)'; this.style.boxShadow='0 6px 20px rgba(0, 123, 255, 0.5)';"
           onmouseout="this.style.transform='scale(1)'; this.style.boxShadow='0 4px 15px rgba(0, 123, 255, 0.3)';"
           onmousedown="this.style.transform='scale(0.98)';">
            <span>  🤖 </span> Abrir o PesquisAI
        </button>
    </a>
</div>
'''

display(HTML(botao_html))

# 🤯 PesquisAI: A ferramenta que seu orientador não te contou! (Só não esqueça de fazer o double-check 😅)

## Declaração de Limitações

O PesquisAI:
- **Não substitui** a revisão por pares nem o julgamento de um pesquisador humano.
- **Não acessa** bases de dados pagas sem integração via skill configurada.
- **Não realiza** coleta primária de dados (entrevistas, experimentos, surveys).
- **Não garante** atualização em tempo real; a disponibilidade dos dados depende das APIs das skills.
---

## 📬 Contato e Suporte

👨‍💻 Criado por **Gustavo Bastos Braga** (UFV) — [gustavo.braga@ufv.br](mailto:gustavo.braga@ufv.br)  
⭐ Código-fonte e contribuições: [GitHub PesquisAI](https://github.com/gustavobraga-byte/PesquisAI)

---

## ⚠️ Atenção: Uso de Dados e Privacidade

*   **Não insira dados pessoais ou sensíveis:** Este ambiente é de teste. Por favor, evite inserir qualquer tipo de informação pessoal identificável, dados confidenciais ou sensíveis durante a utilização do PesquisAI.
*   **Ambiente de Teste:** O PesquisAI está em fase de desenvolvimento e avaliação. Ele é destinado a fins experimentais e de demonstração. Embora busquemos a segurança, a responsabilidade pelo uso de dados é do usuário.